In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
import random

SEED = 42

# Python
random.seed(SEED)

# NumPy
np.random.seed(SEED)

# PyTorch
torch.manual_seed(SEED)

# CUDA
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seeds fixed.")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

In [ ]:
graph_df=pd.read_csv("/Users/007vd/Downloads/DAU/grl-paper/data/embeddings/graph_embeddings.csv")
graph_df

In [ ]:

graph_df["date"]=pd.to_datetime(graph_df["date"])
graph_df=graph_df.sort_values(["date","ticker"]).reset_index(drop=True)

print(graph_df.shape)
graph_df


In [ ]:
embedding_cols=[]
for col in graph_df.columns:
    if col.__contains__("emb_"):
        embedding_cols.append(col)
embedding_cols

In [ ]:
TOP_FEATURES = [
    "trimmed_pce",
    "adj_close",
    "sma_10",
    "rsi_14",
    "personal_savings_rate",
    "macd",
    "open",
    "real_auto_sales",
    "ema_10",
    "cpi",
    "volume",
    "financial_conditions",
    "core_sticky_inflation",
    "recession_probability",
    "ema_5",
    "sma_5",
]
assets = [

    "AAPL",
    "MSFT",
    "NVDA",
    "GOOG",
    "GOOGL",
    "AMZN",
    "TSLA",
    "BRK-B",
    "JNJ",
    "UNH"
]

In [ ]:
DATA_FOLDER=Path("/Users/007vd/Downloads/DAU/grl-paper/data/processed")
stock_data={}
for asset in assets:
    df = pd.read_csv(DATA_FOLDER / f"{asset}_merged.csv")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    keep = ["date"] + TOP_FEATURES
    df = df[keep].dropna()
    stock_data[asset] = df
    
print("loaded dataset:", list(stock_data.keys())) 

In [ ]:
stock_data.get("AAPL")

In [ ]:
ALL_FEATURES=TOP_FEATURES+embedding_cols
common_dates=None

for ticker in assets:
    raw_dates=set(stock_data[ticker]["date"])
    graph_dates = set(graph_df[graph_df["ticker"] == ticker]["date"])

    ticker_dates=raw_dates.intersection(graph_dates)
    if common_dates is None:
        common_dates=ticker_dates
    else:
        common_dates=common_dates.intersection(ticker_dates)
common_dates=sorted(list(common_dates))

print(
    "Number of Common Dates:",
    len(common_dates)
)

In [ ]:
feature_data=[]
future_returns=[]

tickers=sorted(stock_data.keys())
for ticker in tickers:
    raw_df=stock_data[ticker]
    raw_df=raw_df[raw_df["date"].isin(common_dates)]
    raw_df=raw_df.sort_values("date").reset_index(drop=True)

    ticker_graph=graph_df[graph_df["ticker"]==ticker]
    ticker_graph=ticker_graph[ticker_graph["date"].isin(common_dates)]
    ticker_graph=ticker_graph.sort_values("date").reset_index(drop=True)

    raw_features=raw_df[TOP_FEATURES].values
    graph_features=ticker_graph[embedding_cols].values

    combined_features=np.concatenate(
        [raw_features,graph_features],axis=1)
    returns=(raw_df["adj_close"].shift(-1)/raw_df["adj_close"] -1).values

    combined_features = combined_features[:-1]
    returns = returns[:-1]

    feature_data.append(combined_features)
    future_returns.append(returns)

In [ ]:
X=np.stack(feature_data,axis=1)
y=np.stack(future_returns,axis=1)
print("X Shape:", X.shape)
print("y Shape:", y.shape)

In [ ]:
num_samples = X.shape[0]
train_size = int(num_samples * 0.70)
val_size   = int(num_samples * 0.15)
test_size = num_samples - train_size - val_size

In [ ]:
X_train = X[:train_size]
X_val   = X[train_size:train_size + val_size]
X_test  = X[train_size + val_size:]

y_train = y[:train_size]
y_val   = y[train_size:train_size + val_size]
y_test  = y[train_size + val_size:]

print("Train Shape:", X_train.shape)
print("Val Shape:", X_val.shape)
print("Test Shape:", X_test.shape)



In [ ]:
num_assets = X_train.shape[1]
num_features = X_train.shape[2]

X_train=X_train.reshape(X_train.shape[0],num_assets*num_features)
X_val=X_val.reshape(X_val.shape[0],num_assets*num_features)
X_test=X_test.reshape(X_test.shape[0],num_assets*num_features)

print("Flattened Train Shape:", X_train.shape)

In [ ]:
scaler=StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
X_train=torch.tensor(X_train,dtype=torch.float32)
X_test=torch.tensor(X_test,dtype=torch.float32)
X_val=torch.tensor(X_val,dtype=torch.float32)

y_train=torch.tensor(y_train,dtype=torch.float32)
y_test=torch.tensor(y_test,dtype=torch.float32)
y_val=torch.tensor(y_val,dtype=torch.float32)

In [ ]:
BATCH_SIZE=32
train_dataset=TensorDataset(X_train,y_train)
val_dataset=TensorDataset(X_val,y_val)
test_dataset=TensorDataset(X_test,y_test)

train_loader=DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
val_loader=DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
test_loader=DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
class PortfolioANN(nn.Module):
    def __init__(self,input_dim,num_assets):
        super().__init__()
        self.network=nn.Sequential(
            nn.Linear(input_dim,128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, num_assets),
            nn.Softmax(dim=1)
        )
    def forward(self,x):
        return self.network(x)
    

In [ ]:
INPUT_DIM = X_train.shape[1]
NUM_ASSETS = y_train.shape[1]

model=PortfolioANN(
    input_dim=INPUT_DIM,num_assets=NUM_ASSETS
).to(device)

print(model)

In [ ]:
def sharpe_loss(weights,future_returns):

    portfolio_returns=torch.sum(
        weights*future_returns,dim=1
    )

    mean_return=portfolio_returns.mean()
    volatility=portfolio_returns.std()
    sharpe=mean_return/(volatility + 1e-8)
    concentration = torch.mean(weights ** 2)

    loss=-sharpe+0.01*concentration

    return loss


In [ ]:
optimizer=torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

In [ ]:
EPOCHS = 20

train_losses = []

val_losses = []

best_val_loss = float("inf")

best_model_state = None

patience = 8

patience_counter = 0


for epoch in range(EPOCHS):

    # ======================
    # TRAINING
    # ======================

    model.train()

    total_train_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)

        y_batch = y_batch.to(device)

        # Forward pass
        weights = model(X_batch)

        # Compute loss
        loss = sharpe_loss(
            weights,
            y_batch
        )

        # Zero gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # Update weights
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = (
        total_train_loss /
        len(train_loader)
    )

    train_losses.append(avg_train_loss)

    # ======================
    # VALIDATION
    # ======================

    model.eval()

    total_val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)

            y_batch = y_batch.to(device)

            weights = model(X_batch)

            loss = sharpe_loss(
                weights,
                y_batch
            )

            total_val_loss += loss.item()

    avg_val_loss = (
        total_val_loss /
        len(val_loader)
    )

    val_losses.append(avg_val_loss)

    # ======================
    # EARLY STOPPING
    # ======================
    MIN_DELTA = 1e-4

    if avg_val_loss < best_val_loss-MIN_DELTA:

        best_val_loss = avg_val_loss

        best_model_state = model.state_dict()

        patience_counter = 0

    else:

        patience_counter += 1

    # ======================
    # LOGGING
    # ======================

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {avg_train_loss:.10f} "
        f"Val Loss: {avg_val_loss:.10f}"
    )

    # ======================
    # STOPPING CONDITION
    # ======================

    if patience_counter >= patience:

        # print("Early stopping triggered.")

        break

In [ ]:
model.load_state_dict(
    best_model_state
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.show()

In [ ]:
model.eval()
all_weights = []

with torch.no_grad():
    
    for X_batch, _ in test_loader:

        X_batch = X_batch.to(device)

        weights = model(X_batch)

        all_weights.append(
            weights.cpu().numpy()
        )

all_weights = np.vstack(all_weights)

print("Weights Shape:", all_weights.shape)

In [ ]:
test_returns = y_test.cpu().numpy()

portfolio_returns = np.sum(
    all_weights * test_returns,
    axis=1
)

print(portfolio_returns[:5])

In [ ]:
def compute_sharpe(returns):

    mean_return = np.mean(returns)

    volatility = np.std(returns)

    sharpe = (
        np.sqrt(252) * mean_return
    ) / (volatility + 1e-8)

    return sharpe

In [ ]:
sharpe_ratio = compute_sharpe(portfolio_returns)

print("Test Sharpe Ratio:", sharpe_ratio)

In [ ]:
cumulative_returns = np.cumprod(
    1 + portfolio_returns
)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(cumulative_returns)

plt.title("ANN Portfolio Growth")

plt.xlabel("Time")

plt.ylabel("Portfolio Value")

plt.show()

In [ ]:
equal_weights = np.ones_like(test_returns)

equal_weights = (
    equal_weights /
    equal_weights.shape[1]
)

equal_weight_returns = np.sum(
    equal_weights * test_returns,
    axis=1
)

equal_weight_sharpe = compute_sharpe(
    equal_weight_returns
)

print("ANN Sharpe Ratio:", sharpe_ratio)

print("Equal Weight Sharpe Ratio:", equal_weight_sharpe)

equal_weight_cumulative = np.cumprod(
    1 + equal_weight_returns
)
plt.figure(figsize=(12, 6))

plt.plot(
    cumulative_returns,
    label="ANN Portfolio"
)

plt.plot(
    equal_weight_cumulative,
    label="Equal Weight Portfolio"
)

plt.title("ANN vs Equal Weight Benchmark")

plt.xlabel("Time")

plt.ylabel("Portfolio Value")

plt.legend()

plt.show()

In [ ]:
def compute_max_drawdown(cumulative_returns):

    running_max = np.maximum.accumulate(
        cumulative_returns
    )

    drawdowns = (
        cumulative_returns - running_max
    ) / running_max

    max_drawdown = np.min(drawdowns)

    return max_drawdown

In [ ]:
def compute_turnover(weights):

    turnover = np.mean(
        np.sum(
            np.abs(weights[1:] - weights[:-1]),axis=1
        )
    )
    return turnover

In [ ]:
def compute_weight_std(weights):

    return np.std(weights)

In [ ]:
def compute_final_return(cumulative_returns):

    return cumulative_returns[-1] - 1

In [ ]:
def compute_volatility(returns):

    volatility = (
        np.std(returns)
        * np.sqrt(252)
    )

    return volatility

In [ ]:
ann_weights = all_weights

ann_returns = portfolio_returns

ann_cumulative = cumulative_returns

ann_sharpe = sharpe_ratio

In [ ]:
ann_volatility = compute_volatility(
    ann_returns
)

ann_drawdown = compute_max_drawdown(
    ann_cumulative
)

ann_turnover = compute_turnover(
    ann_weights
)

ann_weight_std = compute_weight_std(
    ann_weights
)

ann_final_return = compute_final_return(
    ann_cumulative
)

print("===== ANN GRAPHED METRICS =====")

print("Sharpe:", ann_sharpe)

print("Volatility:", ann_volatility)

print("Max Drawdown:", ann_drawdown)

print("Turnover:", ann_turnover)

print("Weight Std:", ann_weight_std)

print("Final Return:", ann_final_return)

In [ ]:
equal_volatility = compute_volatility(
    equal_weight_returns
)

equal_drawdown = compute_max_drawdown(
    equal_weight_cumulative
)

equal_final_return = compute_final_return(
    equal_weight_cumulative
)

print("===== EQUAL WEIGHT METRICS =====")

print("Sharpe:", equal_weight_sharpe)

print("Volatility:", equal_volatility)

print("Max Drawdown:", equal_drawdown)

print("Final Return:", equal_final_return)

In [ ]:
comparison_df = pd.DataFrame({

    "Model": [
        "Equal Weight",
        
        "ANN_GRAPH"
    ],

    "Sharpe": [
        equal_weight_sharpe,
        
        ann_sharpe
    ],

    "Volatility": [
        equal_volatility,
        
        ann_volatility
    ],

    "Max Drawdown": [
        equal_drawdown,
        
        ann_drawdown
    ],

    "Turnover": [
        0,
        
        ann_turnover
    ],

    "Weight Std": [
        0,
        
        ann_weight_std
    ],

    "Final Return": [
        equal_final_return,
        
        ann_final_return
    ]
})

comparison_df